# HW2 Part 1: Data Cleaning & Validation

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from IPython.display import display

conn = sqlite3.connect('data/data.db')

## 1.1 Load and inspect

In [2]:
raw = pd.read_sql_query('SELECT * FROM ohlc_hw', conn, parse_dates=['ts'])
raw.info()
display(raw.describe())
print('Date range:', raw['ts'].min().date(), 'to', raw['ts'].max().date())
print('Raw ticker values:', raw['ticker'].nunique())
display(raw['ticker'].value_counts().tail(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   ts         397711 non-null  datetime64[ns]
 1   open       397711 non-null  float64       
 2   high       397711 non-null  float64       
 3   low        397711 non-null  float64       
 4   close      397711 non-null  float64       
 5   adj_close  377856 non-null  float64       
 6   volume     397711 non-null  int64         
 7   ticker     397711 non-null  object        
dtypes: datetime64[ns](1), float64(5), int64(1), object(1)
memory usage: 24.3+ MB


,ts,open,high,low,close,adj_close,volume
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,1998-04-17 21:18:45.292485888,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08


Date range: 1962-01-02 to 2024-12-06
Raw ticker values: 150


ticker
<a target="_blank"><span style="color: blue">CRM</span></a>                    129
<a target="_blank"><span style="color: red">CRM</span></a>                     126
<a target="_blank"><span style="color: blue">V</span></a>                      106
<a target="_blank"><span style="color: red">V</span></a>                        97
<a target="_blank"><span style="font-weight: bold; color: red">V</span></a>     96
Name: count, dtype: int64

In [3]:
df = raw.copy()
df['ticker'] = df['ticker'].str.replace(r'<[^>]+>', '', regex=True).str.strip()
df = df.sort_values(['ticker', 'ts']).reset_index(drop=True)
tickers = sorted(df['ticker'].unique())
print(len(tickers), 'tickers:', tickers)

30 tickers: ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


## 1.2 Detect data quality issues

In [4]:
PRICE_COLS = ['open', 'high', 'low', 'close', 'adj_close']


def issues(frame, mask, issue_type, description, severity):
    rows = frame.loc[mask, ['ticker', 'ts']].rename(columns={'ts': 'issue_date'})
    return rows.assign(issue_type=issue_type, description=description, severity=severity)


def detect_html_tickers(raw):
    stripped = raw.assign(ticker=raw['ticker'].str.replace(r'<[^>]+>', '', regex=True).str.strip())
    return issues(stripped, raw['ticker'].str.contains('<'),
                  'html_ticker', 'Ticker wrapped in HTML tags', 'high')


def detect_duplicates(frame):
    return issues(frame, frame.duplicated(['ticker', 'ts']), 'duplicate_row', 'Duplicate ticker/date row', 'high')


def detect_missing_values(frame):
    return pd.concat([issues(frame, frame[col].isna(), 'missing_value', f'{col} is missing', 'medium')
                      for col in PRICE_COLS + ['volume']])


def detect_missing_days(frame):
    calendar = pd.Series(sorted(frame['ts'].unique()))
    rows = []
    for ticker, g in frame.groupby('ticker'):
        expected = calendar[calendar.between(g['ts'].min(), g['ts'].max())]
        missing = expected[~expected.isin(g['ts'])]
        print(f'Ticker {ticker} is missing {len(missing)} trading days between {g["ts"].min().date()} and {g["ts"].max().date()}')
        rows += [{'ticker': ticker, 'issue_date': d, 'issue_type': 'missing_day',
                  'description': 'No row on a market trading day', 'severity': 'medium'} for d in missing]
    return pd.DataFrame(rows, columns=['ticker', 'issue_date', 'issue_type', 'description', 'severity'])


def detect_extreme_returns(frame, col='adj_close', n_std=3):
    returns = frame.groupby('ticker')[col].pct_change(fill_method=None)
    z = (returns - returns.groupby(frame['ticker']).transform('mean')) / returns.groupby(frame['ticker']).transform('std')
    return issues(frame, z.abs() > n_std, 'extreme_return', f'{col} return beyond {n_std} std', 'medium')


def detect_bad_prices(frame):
    return pd.concat([
        issues(frame, frame['high'] < frame['close'], 'high_below_close', 'high < close', 'high'),
        issues(frame, frame['low'] > frame['close'], 'low_above_close', 'low > close', 'high'),
        issues(frame, ~frame['open'].between(frame['low'], frame['high']), 'open_out_of_range', 'open outside [low, high]', 'high'),
        issues(frame, frame[PRICE_COLS].lt(0).any(axis=1), 'negative_price', 'Negative price', 'high'),
        issues(frame, frame['volume'] == 0, 'zero_volume', 'Zero volume (possibly stale)', 'low'),
    ])

In [5]:
dedup = df.drop_duplicates(['ticker', 'ts']).reset_index(drop=True)
quality_report = pd.concat([
    detect_html_tickers(raw),
    detect_duplicates(df),
    detect_missing_values(dedup),
    detect_missing_days(dedup),
    detect_extreme_returns(dedup),
    detect_bad_prices(dedup),
])[['ticker', 'issue_type', 'issue_date', 'description', 'severity']].reset_index(drop=True)
display(quality_report.groupby(['issue_type', 'severity']).size().rename('count').to_frame())
display(quality_report.sample(10, random_state=0))

Ticker AAPL is missing 6 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 25 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 50 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 26 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 25 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 1 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 26 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 25 trading days between 1962-01-02 and 2024-12-06

,,count
issue_type,severity,
duplicate_row,high,29460
extreme_return,medium,7477
html_ticker,high,39936
missing_day,medium,445
missing_value,medium,18374
open_out_of_range,high,5
zero_volume,low,75


,ticker,issue_type,issue_date,description,severity
51005,HON,duplicate_row,1978-03-06,Duplicate ticker/date row,high
79220,JPM,missing_value,2023-01-17,adj_close is missing,medium
12146,AXP,html_ticker,2024-08-14,Ticker wrapped in HTML tags,high
68852,WMT,duplicate_row,1998-01-14,Duplicate ticker/date row,high
62146,NKE,duplicate_row,2017-11-10,Duplicate ticker/date row,high
80100,MCD,missing_value,1971-10-13,adj_close is missing,medium
82101,MRK,missing_value,2007-02-26,adj_close is missing,medium
33730,HON,html_ticker,1972-08-07,Ticker wrapped in HTML tags,high
59230,MMM,duplicate_row,2018-05-04,Duplicate ticker/date row,high
25881,HON,html_ticker,1989-02-02,Ticker wrapped in HTML tags,high


## 1.3 Clean

Decisions:
- Strip tags, drop duplicates and rows with every OHLC price missing. Forward-fill prices within each ticker and fill missing volume with 0.
- **Bad OHLC (`open = 0`)**: option 3. Set open to that day's close and clip it into [low, high]. There is no previous close because these are each ticker's first rows.
- **Extreme returns**: I repair the adjustment factor instead of the price. A factor more than 10% away from its 11-day rolling median is treated as an error and set to NaN, and so is a missing `adj_close`. The factor is then interpolated within the ticker and `adj_close = close * factor`. This keeps the adjustment convention consistent.
- Extreme returns left after the repair are real moves (e.g. crashes or news), so I keep them and flag them.

In [6]:
def clean(frame):
    out = frame.drop_duplicates(['ticker', 'ts']).sort_values(['ticker', 'ts']).reset_index(drop=True)
    out = out.dropna(subset=['open', 'high', 'low', 'close'], how='all')
    out[['open', 'high', 'low', 'close']] = out.groupby('ticker')[['open', 'high', 'low', 'close']].ffill()
    out['volume'] = out['volume'].fillna(0)

    out['open_repaired'] = ~out['open'].between(out['low'], out['high'])
    out.loc[out['open_repaired'], 'open'] = out['close']
    out['open'] = out['open'].clip(out['low'], out['high'])

    factor = out['adj_close'] / out['close']
    median = factor.groupby(out['ticker']).transform(lambda s: s.rolling(11, center=True, min_periods=1).median())
    bad = (factor / median - 1).abs() > 0.1
    out['adj_repaired'] = bad | factor.isna()
    factor = factor.mask(bad).groupby(out['ticker']).transform(lambda s: s.interpolate().ffill().bfill())
    out['adj_close'] = out['close'] * factor

    returns = out.groupby('ticker')['adj_close'].pct_change(fill_method=None)
    z = (returns - returns.groupby(out['ticker']).transform('mean')) / returns.groupby(out['ticker']).transform('std')
    out['extreme_return'] = z.abs() > 3
    out['zero_volume'] = out['volume'] == 0
    return out.reset_index(drop=True)


cleaned = clean(df)
display(cleaned[['open_repaired', 'adj_repaired', 'extreme_return', 'zero_volume']].sum())

open_repaired         5
adj_repaired      28797
extreme_return     5056
zero_volume          75
dtype: int64

## Validation

In [7]:
assert cleaned[PRICE_COLS].notna().all().all()
assert (cleaned['low'] <= cleaned['open']).all() and (cleaned['open'] <= cleaned['high']).all()
assert (cleaned['low'] <= cleaned['close']).all() and (cleaned['close'] <= cleaned['high']).all()
assert (cleaned[PRICE_COLS] > 0).all().all()
assert not cleaned.duplicated(['ticker', 'ts']).any()
print('All checks passed')

print('Before:'); display(raw.describe())
print('After:'); display(cleaned.describe())

All checks passed
Before:


,ts,open,high,low,close,adj_close,volume
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,1998-04-17 21:18:45.292485888,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
min,1962-01-02 00:00:00,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,2000-01-06 00:00:00,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,2012-08-21 00:00:00,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08


After:


,ts,open,high,low,close,adj_close,volume
count,368251,368251.000000,368251.000000,368251.000000,368251.000000,368251.000000,3.682510e+05
mean,1998-04-19 05:03:19.204347008,46.308781,46.771204,45.839768,46.316711,37.274172,3.095142e+07
min,1962-01-02 00:00:00,0.005208,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,1985-05-28 00:00:00,4.210000,4.250000,4.170000,4.210000,1.310000,1.731400e+06
50%,2000-01-07 00:00:00,19.890000,20.090000,19.670000,19.900000,11.460000,4.649600e+06
75%,2012-08-21 00:00:00,57.560000,58.160000,56.990000,57.570000,41.570000,1.102151e+07
max,2024-12-06 00:00:00,604.260000,608.630000,597.880000,605.400000,605.400000,9.276606e+09
std,NaN,67.143005,67.789068,66.489203,67.154690,63.587620,1.266145e+08


In [8]:
# Cross-check against the clean ohlc table that lives in the same database
reference = pd.read_sql_query('SELECT ticker, ts, adj_close FROM ohlc', conn, parse_dates=['ts'])
check = cleaned.merge(reference, on=['ticker', 'ts'], suffixes=('', '_ref'))
error = (check['adj_close'] / check['adj_close_ref'] - 1).abs()
print(f'{(error < 0.01).mean():.4%} of cleaned adj_close within 1% of the reference')

99.9891% of cleaned adj_close within 1% of the reference


In [9]:
cleaned.to_sql('ohlc_hw_cleaned', conn, if_exists='replace', index=False)
print(pd.read_sql_query('SELECT COUNT(*) AS n, COUNT(DISTINCT ticker) AS tickers FROM ohlc_hw_cleaned', conn))
conn.close()

        n  tickers
0  368251       30
